<a href="https://colab.research.google.com/github/jcpd2806/data-mining/blob/main/parte2_data_mining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# New section

In [1]:
from sys import displayhook

import networkx as nx
import matplotlib.pyplot as plt
import os
import pandas as pd
import matplotlib.pyplot as plt
from numpy import dtype
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
import numpy as np
import caja_negra_p2 as caja_negra_p2

df_train = pd.read_csv(os.path.join("train_p2.csv"))
print(df_train['nivel_verificacion'].value_counts())
# display(df_train)

df_test = pd.read_csv(os.path.join("test_p2.csv"))
# display(df_test)



ModuleNotFoundError: No module named 'caja_negra_p2'

In [ ]:
def encoder_df(df, nivel):
    # Label Encoding
    df['niveles_encoded'] = df['nivel_verificacion'].map(nivel)

    # One Hot Encoding
    df = pd.get_dummies(df, columns=['pais', 'dispositivo', 'email_domain'], dtype=int)

    # Seno / Coseno Encoding
    df['fecha_creacion'] = pd.to_datetime(df['fecha_creacion'])
    days = df['fecha_creacion'].dt.dayofyear
    days_in_year = np.where(df['fecha_creacion'].dt.is_leap_year, 366.0, 365.0)

    df['fecha_creacion_sin'] = np.sin(2 * np.pi * days / days_in_year)
    df['fecha_creacion_cos'] = np.cos(2 * np.pi * days / days_in_year)

    # Log Encoding
    df['seguidores'] = np.log1p(df['seguidores'])
    df['siguiendo'] = np.log1p(df['siguiendo'])

    return df

def proportions(df):
    df['likes_per_day'] = df['likes_given_30d'] // 30
    return df

In [ ]:
niveles = {
    'NINGUNO' : 1,
    'VERIFICADO' : 2,
    'TELEFONO' : 3,
    'CORREO' : 4
}

df_train = encoder_df(df_train, niveles)
df_test = encoder_df(df_test, niveles)

display(df_train)
display(df_test)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

url = './train_p2.csv'
df = pd.read_csv(url)

encoded_df = pd.get_dummies(df, columns=['pais', 'email_domain'], dtype=int)

df['ratio_seguidores'] = np.log1p(df['seguidores']) / (np.log1p(df['siguiendo']) + 1)

features = ['variance_post_interval_hr', 'edad_usuario']
df_kmeans = df[features].dropna().copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_kmeans)

scores = {}
for k in range(2, 11):
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_scaled)
    scores[k] = silhouette_score(X_scaled, labels)
best_k = max(scores, key=scores.get)
print('Mejor k:', best_k)

model = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df_kmeans['cluster_kmeans'] = model.fit_predict(X_scaled)

cluster_center = scaler.inverse_transform(model.cluster_centers_)
print(pd.DataFrame(cluster_center, columns=features))

plt.figure(figsize=(8, 6))
plt.scatter(
    df_kmeans['variance_post_interval_hr'],
    df_kmeans['edad_usuario'],
    c=df_kmeans['cluster_kmeans'],
    cmap='viridis',
    s=5,
)
plt.scatter(cluster_center[:, 0], cluster_center[:, 1], c='red', marker='x', s=400)
plt.xlabel('Variance Post Interval (hr)')
plt.ylabel('Edad Usuario')
plt.title(f'KMeans: {best_k} Clusters')
plt.show()
print(df_kmeans)
print(df.head())
df.info()


In [ ]:
from caja_negra_p2 import evaluar_y_generar_submission

dropear = ['user_id', 'fecha_creacion', 'nivel_verificacion']

X_train_engineered = df_train.drop(columns=dropear + ['is_bot'])
X_test_engineered = df_test.drop(columns=dropear)

cv_f1, df_sub = evaluar_y_generar_submission(
    X_train=X_train_engineered,
    y_train=df_train["is_bot"],
    X_test=X_test_engineered,
    test_user_ids=df_test["user_id"],
    output_path="submission_p2.csv"
)